# 🧠 Mini-AlexNet on CIFAR-10

This notebook implements a **scaled-down version of AlexNet** from the 2012 paper *"ImageNet Classification with Deep Convolutional Neural Networks"* by Krizhevsky, Sutskever & Hinton.

### What we keep from the paper:
- Same overall structure: convolutional layers → fully-connected layers → softmax
- ReLU activations (Section 3.1)
- Overlapping max pooling (Section 3.4)
- Dropout on fully-connected layers (Section 4.2)
- Data augmentation: random crops + horizontal flips (Section 4.1)
- SGD with momentum + weight decay (Section 5)
- Learning rate reduction when progress stalls (Section 5)

### What we change:
- **Dataset**: CIFAR-10 instead of ImageNet (10 classes, 32×32 images — much faster to train)
- **Smaller filters and fewer kernels** to fit the smaller image size
- **Fewer parameters** overall (~1M vs 60M in the original)
- **No LRN** (Local Response Normalization — largely obsolete, replaced by BatchNorm)
- **No multi-GPU split** (not needed at this scale)

### CIFAR-10 classes:
✈️ airplane | 🚗 automobile | 🐦 bird | 🐱 cat | 🦌 deer | 🐶 dog | 🐸 frog | 🐴 horse | 🚢 ship | 🚚 truck

In [ ]:
# ── Cell 1: Imports ──────────────────────────────────────────────────────────
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
from torch.optim.lr_scheduler import ReduceLROnPlateau

# Use GPU if available
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')
if device.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')

In [ ]:
# ── Cell 2: Dataset ───────────────────────────────────────────────────────────
# Section 4.1: Data augmentation during training
# - RandomCrop: equivalent to the random 224×224 patches in the paper
# - RandomHorizontalFlip: same as the paper
# - Normalize: equivalent to subtracting mean pixel activity (Section 2)

CIFAR10_MEAN = (0.4914, 0.4822, 0.4465)
CIFAR10_STD  = (0.2470, 0.2435, 0.2616)

train_transform = transforms.Compose([
    transforms.RandomCrop(32, padding=4),       # random crop with padding
    transforms.RandomHorizontalFlip(),           # random horizontal flip
    transforms.ToTensor(),
    transforms.Normalize(CIFAR10_MEAN, CIFAR10_STD),
])

# No augmentation at test time — just normalize
test_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(CIFAR10_MEAN, CIFAR10_STD),
])

train_dataset = torchvision.datasets.CIFAR10(root='./data', train=True,  download=True, transform=train_transform)
test_dataset  = torchvision.datasets.CIFAR10(root='./data', train=False, download=True, transform=test_transform)

train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=128, shuffle=True,  num_workers=2)
test_loader  = torch.utils.data.DataLoader(test_dataset,  batch_size=128, shuffle=False, num_workers=2)

CLASS_NAMES = ['airplane', 'automobile', 'bird', 'cat', 'deer',
               'dog', 'frog', 'horse', 'ship', 'truck']
CLASS_EMOJI = ['✈️', '🚗', '🐦', '🐱', '🦌', '🐶', '🐸', '🐴', '🚢', '🚚']

print(f'Training images : {len(train_dataset):,}')
print(f'Test images     : {len(test_dataset):,}')
print(f'Classes         : {CLASS_NAMES}')

In [ ]:
# ── Cell 3: Visualise some training images ───────────────────────────────────
def denormalize(tensor):
    """Undo normalization so we can display images properly."""
    mean = torch.tensor(CIFAR10_MEAN).view(3,1,1)
    std  = torch.tensor(CIFAR10_STD).view(3,1,1)
    return (tensor * std + mean).clamp(0, 1)

fig, axes = plt.subplots(2, 8, figsize=(16, 5))
fig.suptitle('Sample CIFAR-10 Training Images', fontsize=14, fontweight='bold')
indices = np.random.choice(len(train_dataset), 16, replace=False)
for ax, idx in zip(axes.flat, indices):
    img, label = train_dataset[idx]
    ax.imshow(denormalize(img).permute(1, 2, 0))
    ax.set_title(f'{CLASS_EMOJI[label]} {CLASS_NAMES[label]}', fontsize=9)
    ax.axis('off')
plt.tight_layout()
plt.show()

In [ ]:
# ── Cell 4: Model ─────────────────────────────────────────────────────────────
# Mini-AlexNet: faithful to the paper's structure but scaled down for CIFAR-10.
#
# Original AlexNet (224×224 input):          This version (32×32 input):
#   Conv1: 96 filters, 11×11, stride 4    →  Conv1: 64 filters, 3×3, stride 1
#   Conv2: 256 filters, 5×5              →  Conv2: 128 filters, 3×3
#   Conv3: 384 filters, 3×3              →  Conv3: 256 filters, 3×3
#   Conv4: 384 filters, 3×3              →  Conv4: 256 filters, 3×3
#   Conv5: 256 filters, 3×3              →  Conv5: 128 filters, 3×3
#   FC6: 4096 neurons                    →  FC6: 1024 neurons
#   FC7: 4096 neurons                    →  FC7: 512 neurons
#   FC8: 1000 neurons (softmax)          →  FC8: 10 neurons (softmax)
#
# Why smaller? CIFAR-10 images are 32×32 vs 224×224. Using large filters
# or aggressive strides would destroy the spatial information immediately.

class MiniAlexNet(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()

        # ── Convolutional layers ──────────────────────────────────────────
        # Each block: Conv → ReLU → (optional pooling)
        # Pooling: kernel=3, stride=2 → overlapping pooling from Section 3.4
        self.features = nn.Sequential(
            # Conv1
            nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1),
            nn.ReLU(inplace=True),                    # Section 3.1: ReLU
            nn.MaxPool2d(kernel_size=3, stride=2),    # Section 3.4: overlapping pool

            # Conv2
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=3, stride=2),

            # Conv3 — no pooling between 3, 4, 5 (same as paper)
            nn.Conv2d(128, 256, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),

            # Conv4
            nn.Conv2d(256, 256, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),

            # Conv5
            nn.Conv2d(256, 128, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=3, stride=2),
        )

        # ── Fully-connected layers ────────────────────────────────────────
        # Dropout (p=0.5) on first two FC layers — Section 4.2
        self.classifier = nn.Sequential(
            nn.Dropout(p=0.5),                        # Section 4.2: dropout
            nn.Linear(128 * 2 * 2, 1024),
            nn.ReLU(inplace=True),

            nn.Dropout(p=0.5),                        # Section 4.2: dropout
            nn.Linear(1024, 512),
            nn.ReLU(inplace=True),

            nn.Linear(512, num_classes),              # Final: 10-way output
        )

    def forward(self, x):
        x = self.features(x)
        x = x.view(x.size(0), -1)   # flatten
        x = self.classifier(x)
        return x  # raw logits; CrossEntropyLoss applies softmax internally


model = MiniAlexNet(num_classes=10).to(device)

# Count parameters
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Total trainable parameters: {total_params:,}')
print(f'(Original AlexNet had ~60,000,000)')
print()
print(model)

In [ ]:
# ── Cell 5: Training setup ────────────────────────────────────────────────────
# Section 5:
# - SGD with momentum 0.9  ✓
# - Weight decay 0.0005    ✓
# - Learning rate starts at 0.01, reduced when validation stops improving ✓

criterion = nn.CrossEntropyLoss()

optimizer = optim.SGD(
    model.parameters(),
    lr=0.01,           # Section 5: initial learning rate
    momentum=0.9,      # Section 5: momentum
    weight_decay=5e-4  # Section 5: weight decay (0.0005)
)

# Section 5: divide LR by 10 when validation error stops improving
scheduler = ReduceLROnPlateau(optimizer, mode='min', factor=0.1, patience=5, verbose=True)

print('Optimizer:', optimizer)
print('Scheduler: ReduceLROnPlateau (÷10 after 5 epochs without improvement)')

In [ ]:
# ── Cell 6: Training loop ─────────────────────────────────────────────────────
NUM_EPOCHS = 30

history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}

def run_epoch(loader, training=True):
    if training:
        model.train()
    else:
        model.eval()

    total_loss, correct, total = 0.0, 0, 0
    context = torch.enable_grad() if training else torch.no_grad()

    with context:
        for inputs, labels in loader:
            inputs, labels = inputs.to(device), labels.to(device)
            if training:
                optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            if training:
                loss.backward()
                optimizer.step()
            total_loss += loss.item() * inputs.size(0)
            _, predicted = outputs.max(1)
            correct += predicted.eq(labels).sum().item()
            total += labels.size(0)

    return total_loss / total, 100.0 * correct / total


print(f'Training for {NUM_EPOCHS} epochs on {device}...')
print(f'{"Epoch":>6} | {"Train Loss":>10} | {"Train Acc":>9} | {"Val Loss":>8} | {"Val Acc":>8} | {"LR":>8}')
print('-' * 65)

for epoch in range(1, NUM_EPOCHS + 1):
    train_loss, train_acc = run_epoch(train_loader, training=True)
    val_loss,   val_acc   = run_epoch(test_loader,  training=False)
    scheduler.step(val_loss)

    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)

    current_lr = optimizer.param_groups[0]['lr']
    print(f'{epoch:>6} | {train_loss:>10.4f} | {train_acc:>8.2f}% | {val_loss:>8.4f} | {val_acc:>7.2f}% | {current_lr:>8.6f}')

print(f'\nBest validation accuracy: {max(history["val_acc"]):.2f}%')

In [ ]:
# ── Cell 7: Training curves ───────────────────────────────────────────────────
epochs = range(1, NUM_EPOCHS + 1)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Mini-AlexNet Training on CIFAR-10', fontsize=14, fontweight='bold')

# Loss
ax1.plot(epochs, history['train_loss'], 'b-o', markersize=3, label='Train loss')
ax1.plot(epochs, history['val_loss'],   'r-o', markersize=3, label='Val loss')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.set_title('Loss over Training')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Accuracy
ax2.plot(epochs, history['train_acc'], 'b-o', markersize=3, label='Train accuracy')
ax2.plot(epochs, history['val_acc'],   'r-o', markersize=3, label='Val accuracy')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Accuracy (%)')
ax2.set_title('Accuracy over Training')
ax2.legend()
ax2.grid(True, alpha=0.3)

# Annotate best val accuracy
best_epoch = np.argmax(history['val_acc']) + 1
best_acc   = max(history['val_acc'])
ax2.annotate(f'Best: {best_acc:.1f}%',
             xy=(best_epoch, best_acc),
             xytext=(best_epoch + 1, best_acc - 5),
             arrowprops=dict(arrowstyle='->', color='green'),
             color='green', fontweight='bold')

plt.tight_layout()
plt.show()

# Overfitting check
gap = history['train_acc'][-1] - history['val_acc'][-1]
print(f'\nFinal train accuracy : {history["train_acc"][-1]:.2f}%')
print(f'Final val accuracy   : {history["val_acc"][-1]:.2f}%')
print(f'Train/val gap        : {gap:.2f}% ', end='')
if gap > 15:
    print('⚠️  (some overfitting — try more dropout or augmentation)')
elif gap > 5:
    print('✅  (modest gap, normal for this architecture)')
else:
    print('🎉  (very little overfitting!)')

In [ ]:
# ── Cell 8: Per-class accuracy ────────────────────────────────────────────────
model.eval()
class_correct = [0] * 10
class_total   = [0] * 10

with torch.no_grad():
    for inputs, labels in test_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        outputs = model(inputs)
        _, predicted = outputs.max(1)
        for label, pred in zip(labels, predicted):
            class_correct[label] += (label == pred).item()
            class_total[label] += 1

class_acc = [100.0 * class_correct[i] / class_total[i] for i in range(10)]

# Bar chart
fig, ax = plt.subplots(figsize=(12, 5))
colors = plt.cm.RdYlGn([a / 100 for a in class_acc])
bars = ax.bar(range(10), class_acc, color=colors, edgecolor='white', linewidth=1.5)

ax.set_xticks(range(10))
ax.set_xticklabels([f'{CLASS_EMOJI[i]}\n{CLASS_NAMES[i]}' for i in range(10)], fontsize=11)
ax.set_ylabel('Accuracy (%)')
ax.set_title('Per-Class Accuracy on Test Set', fontsize=14, fontweight='bold')
ax.set_ylim(0, 105)
ax.axhline(y=sum(class_acc)/10, color='navy', linestyle='--', alpha=0.6, label=f'Average: {sum(class_acc)/10:.1f}%')
ax.legend()
ax.grid(axis='y', alpha=0.3)

for bar, acc in zip(bars, class_acc):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
            f'{acc:.1f}%', ha='center', va='bottom', fontsize=9, fontweight='bold')

plt.tight_layout()
plt.show()

print('\nPer-class breakdown:')
for i in range(10):
    bar = '█' * int(class_acc[i] / 5)
    print(f'  {CLASS_EMOJI[i]} {CLASS_NAMES[i]:12s}: {class_acc[i]:5.1f}%  {bar}')

In [ ]:
# ── Cell 9: Visual predictions on random test images ─────────────────────────
# Shows 16 test images with the model's top-5 predictions,
# mimicking Figure 4 in the paper.

model.eval()
indices = np.random.choice(len(test_dataset), 16, replace=False)

fig, axes = plt.subplots(4, 4, figsize=(16, 16))
fig.suptitle('Model Predictions on Test Images\n(Green border = correct, Red = wrong)',
             fontsize=14, fontweight='bold')

with torch.no_grad():
    for ax, idx in zip(axes.flat, indices):
        img_tensor, true_label = test_dataset[idx]
        output = model(img_tensor.unsqueeze(0).to(device))
        probs  = torch.softmax(output, dim=1).squeeze().cpu().numpy()
        top5_idx  = probs.argsort()[::-1][:5]
        top5_prob = probs[top5_idx]
        predicted = top5_idx[0]
        correct   = (predicted == true_label)

        # Show image
        ax.imshow(denormalize(img_tensor).permute(1, 2, 0))

        # Border colour: green if correct, red if wrong
        border_color = '#2ecc71' if correct else '#e74c3c'
        for spine in ax.spines.values():
            spine.set_edgecolor(border_color)
            spine.set_linewidth(4)

        # Build label text: top-5 predictions with probabilities
        label_lines = [f'✅ True: {CLASS_EMOJI[true_label]} {CLASS_NAMES[true_label]}']
        label_lines.append('─' * 24)
        for rank, (ci, prob) in enumerate(zip(top5_idx, top5_prob)):
            marker = '👉' if rank == 0 else f' {rank+1}.'
            line = f'{marker} {CLASS_EMOJI[ci]} {CLASS_NAMES[ci]}: {prob*100:.1f}%'
            label_lines.append(line)

        ax.set_title('\n'.join(label_lines), fontsize=7.5,
                     loc='left', pad=4,
                     color='black',
                     fontfamily='monospace')
        ax.set_xticks([])
        ax.set_yticks([])

plt.tight_layout()
plt.show()

In [ ]:
# ── Cell 10: Confusion matrix ─────────────────────────────────────────────────
# Shows WHICH classes the model confuses with each other.
# A good model has bright diagonal (correct predictions) and dark off-diagonal.

from sklearn.metrics import confusion_matrix
import itertools

all_preds, all_labels = [], []
model.eval()
with torch.no_grad():
    for inputs, labels in test_loader:
        outputs = model(inputs.to(device))
        _, predicted = outputs.max(1)
        all_preds.extend(predicted.cpu().numpy())
        all_labels.extend(labels.numpy())

cm = confusion_matrix(all_labels, all_preds)
cm_norm = cm.astype('float') / cm.sum(axis=1, keepdims=True)  # normalise rows

fig, ax = plt.subplots(figsize=(10, 8))
im = ax.imshow(cm_norm, interpolation='nearest', cmap='Blues')
plt.colorbar(im, ax=ax, fraction=0.046)
ax.set_title('Confusion Matrix (normalised)\nBright diagonal = good!',
             fontsize=13, fontweight='bold')

tick_labels = [f'{CLASS_EMOJI[i]} {CLASS_NAMES[i]}' for i in range(10)]
ax.set_xticks(range(10))
ax.set_yticks(range(10))
ax.set_xticklabels(tick_labels, rotation=45, ha='right', fontsize=9)
ax.set_yticklabels(tick_labels, fontsize=9)
ax.set_ylabel('True label')
ax.set_xlabel('Predicted label')

# Annotate each cell
thresh = cm_norm.max() / 2
for i, j in itertools.product(range(10), range(10)):
    ax.text(j, i, f'{cm_norm[i,j]:.2f}',
            ha='center', va='center', fontsize=7,
            color='white' if cm_norm[i,j] > thresh else 'black')

plt.tight_layout()
plt.show()

# Print the most common confusions
print('\nTop confusions (what the model mistakes for what):')
off_diag = [(cm[i,j], i, j) for i in range(10) for j in range(10) if i != j]
off_diag.sort(reverse=True)
for count, true, pred in off_diag[:5]:
    print(f'  {CLASS_EMOJI[true]} {CLASS_NAMES[true]:12s} → mistaken for {CLASS_EMOJI[pred]} {CLASS_NAMES[pred]:12s}  ({count} times)')

## 📚 What to notice

**Training curves (Cell 7)**  
If train accuracy is much higher than val accuracy, that's overfitting — exactly the problem Section 4 of the paper addresses. You should see the gap stay fairly modest thanks to dropout and data augmentation.

**Per-class accuracy (Cell 8)**  
Some classes are harder than others. Cats and dogs tend to be confused (they look similar), while ships and airplanes tend to score well (distinctive silhouettes). This mirrors what the paper shows in Figure 4.

**Visual predictions (Cell 9)**  
Notice that even wrong predictions are often *reasonable* — mistaking a cat for a dog, or a truck for an automobile. This is exactly the behaviour highlighted in Section 6.1 of the paper.

**Confusion matrix (Cell 10)**  
The bright diagonal means the model is mostly right. The brightest off-diagonal entries reveal the most common mistakes — look for cat↔dog and automobile↔truck confusions.

## 🔧 Things to experiment with
- Remove dropout and retrain — does overfitting get worse?
- Remove data augmentation — does validation accuracy drop?
- Add more convolutional layers — does it improve?
- Try replacing MaxPool with BatchNorm — a taste of what comes after AlexNet